In [ ]:
import pandas as pd
import os
from pathlib import Path

def load_raw_data(filepath):
    return pd.read_csv(filepath)

def clean_nav_dataset(dataframe):
    dataframe["date"] = pd.to_datetime(dataframe["date"], errors="coerce")
    
    dataframe = dataframe.sort_values(["amfi_code", "date"])
    
    dataframe["nav"] = dataframe.groupby("amfi_code")["nav"].ffill()
    
    dataframe = dataframe.drop_duplicates()
    dataframe = dataframe[dataframe["nav"] > 0]
    dataframe = dataframe.dropna(subset=["date"])
    
    return dataframe

def save_cleaned_data(dataframe, output_dir, filename):
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, filename)
    dataframe.to_csv(full_path, index=False)
    return full_path

def generate_report(original, cleaned):
    print(f"Original shape: {original.shape}")
    print(f"Cleaned shape: {cleaned.shape}")
    print(f"Rows removed: {original.shape[0] - cleaned.shape[0]}")
    print(f"\nMissing values:\n{cleaned.isnull().sum()}")
    print(f"\nDuplicate count: {cleaned.duplicated().sum()}")


INPUT_FILE = "data/raw/02_nav_history.csv"
OUTPUT_DIR = "data/processed"
OUTPUT_FILE = "clean_nav_history.csv"

raw_nav = load_raw_data(INPUT_FILE)

processed_nav = clean_nav_dataset(raw_nav)

generate_report(raw_nav, processed_nav)

saved_path = save_cleaned_data(processed_nav, OUTPUT_DIR, OUTPUT_FILE)
print(f"\nSaved: {saved_path}")

print(os.listdir(OUTPUT_DIR))

Original shape: (46000, 3)
Cleaned shape: (46000, 3)
Rows removed: 0

Missing values:
amfi_code    0
date         0
nav          0
dtype: int64

Duplicate count: 0

Saved: data/processed\clean_nav_history.csv
['clean_nav_history.csv']
